# Initial data exploration

Getting familiar with the IBM HR dataset before running the main analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('../data/ibm_hr_attrition.csv', sep=None, engine='python')
print(df.shape)
df.head()

In [ ]:
# class balance -- want to know how imbalanced this is
print(df['Attrition'].value_counts())
print(df['Attrition'].value_counts(normalize=True))

In [ ]:
# age distribution -- split point for fairness analysis
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].hist(df['Age'], bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Age distribution (all employees)')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')

# attrition by age group
df['age_group'] = (df['Age'] < 40).map({True: 'Under 40', False: 'Over 40'})
attr_by_age = df.groupby('age_group')['Attrition'].apply(lambda x: (x == 'Yes').mean())
attr_by_age.plot(kind='bar', ax=axes[1], color=['#fc8d59', '#d7191c'])
axes[1].set_title('Attrition rate by age group')
axes[1].set_ylabel('Attrition rate')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()
print(attr_by_age)

## Notes

- Attrition rate is about 16% -- pretty imbalanced
- Under-40 group has higher raw attrition rate, which probably explains the age EOD
- Might be worth checking overtime vs attrition next (heard that's a strong predictor)

In [ ]:
# quick look at which features correlate most with attrition
df_enc = df.copy()
df_enc['Attrition_bin'] = (df_enc['Attrition'] == 'Yes').astype(int)
numeric_cols = df_enc.select_dtypes(include='number').columns
correlations = df_enc[numeric_cols].corr()['Attrition_bin'].abs().sort_values(ascending=False)
print(correlations.head(15))

## Next steps

- Run the main calibration_analysis.py pipeline
- Look at reliability diagrams more carefully once I have the actual model outputs
- Figure out the right way to do post-hoc calibration (CalibratedClassifierCV seems to have changed)